# Graphs

**Geometry type:** `graph`

Arbitrary vertex–edge graphs embedded in 3-D space. Suitable for vascular
networks, synaptic connectivity graphs, and any structure with cycles.

1. Generate a synthetic vascular network
2. Write (undirected, not a tree)
3. Open lazily and inspect
4. Read all nodes and edges
5. Spatial bounding-box query
6. Directed graph example
7. Validate


In [ ]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import ZarrVectorStore
from zarr_vectors.types.graphs import write_graph, read_graph
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zvf_examples_")
STORE  = os.path.join(_tmpdir, "vessels.zarrvectors")
STORE2 = os.path.join(_tmpdir, "directed.zarrvectors")
print("Stores:", STORE, STORE2)


## 1 · Generate a synthetic vascular network graph

In [ ]:
rng = np.random.default_rng(2)
n_nodes = 3_000

# Nodes scattered in a 600³ µm volume
positions = rng.uniform(0, 600, (n_nodes, 3)).astype(np.float32)

# Edges: connect each node to its ~4 nearest neighbours (simplified)
# (In practice, use a real graph from a segmentation pipeline)
from scipy.spatial import KDTree
tree = KDTree(positions)
_, nbrs = tree.query(positions, k=5)   # k=1 is self; skip it
src = np.repeat(np.arange(n_nodes, dtype=np.int32), 4)
dst = nbrs[:, 1:].flatten().astype(np.int32)

# Remove self-loops and duplicates
mask  = src != dst
edges = np.column_stack([src[mask], dst[mask]])
# Keep canonical undirected form [min, max]
edges = np.sort(edges, axis=1)
edges = np.unique(edges, axis=0).astype(np.int32)

# Per-node attributes
diameter = rng.uniform(1, 20, n_nodes).astype(np.float32)   # µm
flow     = rng.uniform(0, 1, n_nodes).astype(np.float32)    # normalised

print(f"nodes : {n_nodes:,}")
print(f"edges : {len(edges):,}")
print(f"mean degree : {2 * len(edges) / n_nodes:.1f}")


## 2 · Write undirected graph

In [ ]:
write_graph(
    STORE,
    positions=positions,
    edges=edges,
    chunk_shape=(150.0, 150.0, 150.0),
    bin_shape=(37.5, 37.5, 37.5),
    is_directed=False,
    is_tree=False,
    vertex_attributes={
        "diameter": diameter,
        "flow":     flow,
    },
    coordinate_system="RAS",
    axis_units="micrometer",
)
print("Write complete.")


## 3 · Open lazily and inspect

In [ ]:
store = ZarrVectorStore(STORE)
print(f"geometry_type : {store.geometry_type}")
print(f"spatial_dims  : {store.spatial_dims}")
print(f"chunk_shape   : {store.chunk_shape}")
print(f"vertex_count  : {store.vertex_count(0):,}")


## 4 · Read all nodes and edges

In [ ]:
result = read_graph(STORE)
print(f"node_count    : {result['node_count']:,}")
print(f"edge_count    : {result['edge_count']:,}")
print(f"positions     : {result['positions'].shape}")
print(f"edges         : {result['edges'].shape}")
print(f"diameter range: [{result['attributes']['diameter'].min():.1f}, "
      f"{result['attributes']['diameter'].max():.1f}] µm")


## 5 · Spatial bounding-box query

In [ ]:
lo = np.array([200.0, 200.0, 200.0])
hi = np.array([400.0, 400.0, 400.0])

bbox_result = read_graph(STORE, bbox=(lo, hi))
print(f"Nodes in 200³ µm bbox : {bbox_result['node_count']:,}")
print(f"Edges (both ends in bbox): {bbox_result['edge_count']:,}")

# With boundary edges (one endpoint outside bbox)
bbox_boundary = read_graph(STORE, bbox=(lo, hi), include_boundary_edges=True)
print(f"Edges (incl. boundary)   : {bbox_boundary['edge_count']:,}")


## 6 · Directed graph example

In [ ]:
# Directed graph: edges represent directed flow (small network)
rng2   = np.random.default_rng(10)
n_dir  = 200
pos_dir = rng2.uniform(0, 200, (n_dir, 3)).astype(np.float32)
src_dir = rng2.integers(0, n_dir, 300).astype(np.int32)
dst_dir = rng2.integers(0, n_dir, 300).astype(np.int32)
mask_d  = src_dir != dst_dir
edges_dir = np.column_stack([src_dir[mask_d], dst_dir[mask_d]]).astype(np.int32)

write_graph(
    STORE2,
    positions=pos_dir,
    edges=edges_dir,
    chunk_shape=(100., 100., 100.),
    bin_shape=(25., 25., 25.),
    is_directed=True,   # direction is [source, destination]
)
result_dir = read_graph(STORE2)
print(f"Directed graph: {result_dir['node_count']} nodes, "
      f"{result_dir['edge_count']} directed edges")


## 7 · Validate

In [ ]:
rv = validate(STORE, level=3)
print(rv.summary())


## Summary

| Step | API |
|------|-----|
| Write | `write_graph(path, positions, edges, is_directed, is_tree, vertex_attributes)` |
| Read all | `read_graph(path)` |
| Bbox query | `read_graph(path, bbox=(lo, hi))` |
| Bbox + boundary | `read_graph(path, bbox=..., include_boundary_edges=True)` |
